# Authentication & Configuration

In [3]:
import os

# --- GCP Configuration ---
from google.colab import userdata

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]

os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

# --- Authentication ---
from google.colab import auth
auth.authenticate_user()

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

Settings loaded from Colab Secrets.


# Helper functions

In [6]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
RAW = f"{PROJECT_ID}.tennis_raw.wta_matches"
CLEAN = f"{PROJECT_ID}.tennis_clean.matches"

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.width", 200)

# Examining score feature
In additiona to 'normal' scores, also retirement (one player forfeits during match), walkover (one player forfeits before the match) and default (one player loses via disqualification etc)

In [7]:
run(f"""
SELECT COALESCE(REGEXP_EXTRACT(score, r'([A-Za-z]+(?:/[A-Za-z]+)?)'), '(normal score)') AS tag,
       COUNT(*) AS matches
FROM `{RAW}`
WHERE tourney_level IN ('G','PM','P','I')
GROUP BY tag ORDER BY matches DESC
""")

,tag,matches
0,(normal score),37542
1,RET,1173
2,W/O,261
3,DEF,22


# Building the table
Note bytes processed - indicator of cost

In [24]:
INT_COLS = ["draw_size", "best_of", "match_num", "winner_id", "winner_seed", "winner_ht",
            "loser_id", "loser_seed", "loser_ht", "minutes",
            "w_ace", "w_df", "w_svpt", "w_1stIn", "w_1stWon", "w_2ndWon", "w_SvGms", "w_bpSaved", "w_bpFaced",
            "l_ace", "l_df", "l_svpt", "l_1stIn", "l_1stWon", "l_2ndWon", "l_SvGms", "l_bpSaved", "l_bpFaced",
            "winner_rank", "winner_rank_points", "loser_rank", "loser_rank_points"]
FLOAT_COLS = ["winner_age", "loser_age"]
TEXT_COLS = ["winner_name", "winner_hand", "winner_ioc", "loser_name", "loser_hand", "loser_ioc", "score", "round"]
UPPER_COLS = []          # was ["winner_entry", "loser_entry"]; entry is handled below

select_parts = (
    [f"SAFE_CAST(NULLIF(TRIM({c}), '') AS INT64) AS {c}" for c in INT_COLS]
    + [f"SAFE_CAST(NULLIF(TRIM({c}), '') AS FLOAT64) AS {c}" for c in FLOAT_COLS]
    + [f"NULLIF(TRIM({c}), '') AS {c}" for c in TEXT_COLS]
    + [f"UPPER(NULLIF(TRIM({c}), '')) AS {c}" for c in UPPER_COLS]
)

#keeps an existing entry value if there is one, and otherwise takes a non-numeric seed value as the entry code. The seed columns still convert with SAFE_CAST, so those 97 seeds stay NULL, which is right, because those players weren't seeded.
def entry_expr(side):
    seed = f"NULLIF(TRIM({side}_seed), '')"
    return f"""COALESCE(
        UPPER(NULLIF(TRIM({side}_entry), '')),
        IF(SAFE_CAST({seed} AS INT64) IS NULL, UPPER({seed}), NULL)
      ) AS {side}_entry"""

select_parts += [entry_expr("winner"), entry_expr("loser")]

sql = f"""
CREATE OR REPLACE TABLE `{CLEAN}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'Typed WTA matches, levels G/PM/P/I. Built from tennis_raw.wta_matches.')
AS
SELECT
  CONCAT(tourney_id, '-', match_num) AS match_id,
  tourney_id,
  tourney_name,
  LOWER(TRIM(tourney_name)) AS name_key,
  PARSE_DATE('%Y%m%d', tourney_date) AS tourney_date,
  EXTRACT(YEAR FROM PARSE_DATE('%Y%m%d', tourney_date)) AS season,
  tourney_level,
  LOWER(NULLIF(TRIM(surface), '')) AS surface,
  REGEXP_CONTAINS(COALESCE(score, ''), r'W/O') AS is_walkover,
  REGEXP_CONTAINS(COALESCE(score, ''), r'RET') AS is_retirement,
  REGEXP_CONTAINS(COALESCE(score, ''), r'DEF') AS is_default,
  {", ".join(select_parts)}
FROM `{RAW}`
WHERE tourney_level IN ('G', 'PM', 'P', 'I')
  AND tourney_id NOT IN (SELECT DISTINCT tourney_id FROM `{RAW}` WHERE round = 'RR')
"""


job = bq.query(sql)
job.result()
print("Built", CLEAN, "| bytes processed:", f"{job.total_bytes_processed:,}")

Built tennis-project-510110.tennis_clean.matches | bytes processed: 10,612,510


In [26]:
# 1. Row count and uniqueness
print(run(f"SELECT COUNT(*) AS rows_total, COUNT(DISTINCT match_id) AS unique_match_ids FROM `{CLEAN}`"))

# 2. Did any real value fail to convert? (raw had a value, typed table has NULL)
cols = INT_COLS + FLOAT_COLS
bad = run(f"""
SELECT {", ".join(f"COUNTIF(NULLIF(TRIM(r.{c}), '') IS NOT NULL AND m.{c} IS NULL) AS {c}" for c in cols)},
       COUNT(*) AS joined_rows
FROM `{RAW}` r
JOIN `{CLEAN}` m ON r.tourney_id = m.tourney_id AND SAFE_CAST(r.match_num AS INT64) = m.match_num
""").T
print(bad[bad[0] > 0])     # shows joined_rows, plus any column that lost values

# 3. Labels
print(run(f"SELECT tourney_level, surface, COUNT(*) AS matches FROM `{CLEAN}` GROUP BY 1, 2 ORDER BY 1, 2"))
print(run(f"SELECT season, COUNT(*) AS matches FROM `{CLEAN}` GROUP BY 1 ORDER BY 1"))
print(run(f"SELECT COUNTIF(is_walkover) AS walkovers, COUNTIF(is_retirement) AS retirements, COUNTIF(is_default) AS defaults FROM `{CLEAN}`"))

   rows_total  unique_match_ids
0       38860             38860
                 0
winner_seed     25
loser_seed      72
joined_rows  38860
   tourney_level surface  matches
0              G    clay     2159
1              G   grass     1905
2              G    hard     4191
3              I    clay     4693
4              I   grass     1321
5              I    hard     8701
6              P  carpet      141
7              P    clay     2309
8              P   grass      965
9              P    hard     7160
10            PM    clay     1378
11            PM    hard     3937
    season  matches
0     2010     2425
1     2011     2445
2     2012     2468
3     2013     2412
4     2014     2355
5     2015     2491
6     2016     2523
7     2017     2532
8     2018     2530
9     2019     2382
10    2020     1086
11    2021     2184
12    2022     2354
13    2023     2460
14    2024     2553
15    2025     2412
16    2026     1248
   walkovers  retirements  defaults
0        261         1

check the entry column:

In [14]:
run(f"SELECT winner_entry, COUNT(*) AS n FROM `{CLEAN}` GROUP BY 1 ORDER BY n DESC")

,winner_entry,n
0,None,32932
1,Q,3600
2,WC,1903
3,LL,497
4,SR,29
5,SE,22
6,ALT,9
7,PR,6


The WINDOWS dictionary is the record of the assumption, in code, so you can change it and rebuild.

In [27]:
# Day offsets from tourney_date: (earliest, latest). Typical-schedule assumptions, wide on purpose.
WINDOWS = {
    "SLAM":      {"R128": (0, 1), "R64": (2, 3), "R32": (4, 5), "R16": (6, 7), "QF": (8, 9), "SF": (9, 11), "F": (11, 13)},
    "TWO_WEEK":  {"R128": (0, 4), "R64": (2, 6), "R32": (4, 8), "R16": (7, 10), "QF": (8, 11), "SF": (9, 12), "F": (11, 13)},
    "LONG_WEEK": {"R64": (0, 2), "R32": (1, 3), "R16": (2, 4), "QF": (3, 5), "SF": (4, 6), "F": (5, 7)},
    "ONE_WEEK":  {"R32": (0, 2), "R16": (1, 3), "QF": (2, 4), "SF": (3, 5), "F": (5, 6)},
}
RR_WINDOW = (0, 6)    # round robin: we only know it is somewhere in the week

rows = []
for fmt, rounds in WINDOWS.items():
    for rnd, (lo, hi) in rounds.items():
        rows.append(f"SELECT '{fmt}' AS fmt, '{rnd}' AS round, {lo} AS day_min, {hi} AS day_max")
    rows.append(f"SELECT '{fmt}', 'RR', {RR_WINDOW[0]}, {RR_WINDOW[1]}")
windows_sql = "\nUNION ALL\n".join(rows)

DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"
sql = f"""
CREATE OR REPLACE TABLE `{DATED}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'tennis_clean.matches plus estimated match-day windows (event format and round).')
AS
WITH event_format AS (
  SELECT tourney_id,
    CASE
      WHEN ANY_VALUE(tourney_level) = 'G' THEN 'SLAM'
      WHEN LOGICAL_OR(round = 'R128') THEN 'TWO_WEEK'
      WHEN LOGICAL_OR(round = 'R64') THEN 'LONG_WEEK'
      ELSE 'ONE_WEEK'
    END AS fmt
  FROM `{CLEAN}` GROUP BY tourney_id
),
windows AS ({windows_sql})
SELECT m.*, f.fmt AS event_format, w.day_min, w.day_max,
       DATE_ADD(m.tourney_date, INTERVAL w.day_min DAY) AS match_date_min,
       DATE_ADD(m.tourney_date, INTERVAL w.day_max DAY) AS match_date_max
FROM `{CLEAN}` m
JOIN event_format f USING (tourney_id)
LEFT JOIN windows w ON w.fmt = f.fmt AND w.round = m.round
"""
job = bq.query(sql); job.result()
print("Built", DATED, "| bytes processed:", f"{job.total_bytes_processed:,}")

Built tennis-project-510110.tennis_clean.matches_dated | bytes processed: 15,386,846


In [28]:
# 1. Every match got a window; how many events per format
print(run(f"""SELECT event_format, COUNT(DISTINCT tourney_id) AS events, COUNT(*) AS matches,
                     COUNTIF(day_min IS NULL) AS no_window FROM `{DATED}` GROUP BY 1 ORDER BY 1"""))

# 2. What weekday do tournaments start on? (assumed: mostly Monday)
print(run(f"""SELECT FORMAT_DATE('%a', tourney_date) AS weekday, COUNT(DISTINCT tourney_id) AS events
              FROM `{DATED}` GROUP BY 1 ORDER BY events DESC"""))

# 3. Reality check on Slam finals (I know the 2019 women's finals were all on Saturdays:
#    AO 26 Jan, RG 8 Jun, Wimbledon 13 Jul, US Open 7 Sep)
print(run(f"""SELECT name_key, tourney_date, FORMAT_DATE('%a', tourney_date) AS start_day,
                     match_date_min, match_date_max
              FROM `{DATED}` WHERE season = 2019 AND tourney_level = 'G' AND round = 'F'
              ORDER BY tourney_date"""))

# 4. Do any windows reach beyond the weather we fetched?
print(run(f"""SELECT COUNT(DISTINCT m.tourney_id) AS events_short_of_weather
              FROM `{DATED}` m JOIN `{PROJECT_ID}.tennis_clean.tournament_editions` e USING (tourney_id)
              WHERE m.match_date_max > e.end_date"""))

  event_format  events  matches  no_window
0    LONG_WEEK     138     7591          0
1     ONE_WEEK     625    18834          0
2         SLAM      65     8255          0
3     TWO_WEEK      44     4180          0
  weekday  events
0     Mon     838
1     Sun      16
2     Wed       8
3     Tue       6
4     Thu       3
5     Sat       1
          name_key tourney_date start_day match_date_min match_date_max
0  australian open   2019-01-14       Mon     2019-01-25     2019-01-27
1    roland garros   2019-05-27       Mon     2019-06-07     2019-06-09
2        wimbledon   2019-07-01       Mon     2019-07-12     2019-07-14
3          us open   2019-08-26       Mon     2019-09-06     2019-09-08
   events_short_of_weather
0                        6


Estimating match days depends on how rounds are labelled, so we run:

In [15]:
print(run(f"SELECT tourney_level, round, COUNT(*) AS matches, COUNT(DISTINCT tourney_id) AS tournaments FROM `{CLEAN}` GROUP BY 1, 2 ORDER BY 1, matches DESC"))
print(run(f"SELECT tourney_level, draw_size, COUNT(DISTINCT tourney_id) AS tournaments FROM `{CLEAN}` GROUP BY 1, 2 ORDER BY 1, 2"))

   tourney_level round  matches  tournaments
0              G  R128     4160           65
1              G   R64     2080           65
2              G   R32     1040           65
3              G   R16      520           65
4              G    QF      260           65
5              G    SF      130           65
6              G     F       65           65
7              I   R32     7482          471
8              I   R16     3760          470
9              I    QF     1896          474
10             I    SF      952          475
11             I     F      476          475
12             I   R64      176            8
13             I    RR       96            4
14             P   R32     3809          269
15             P   R64     2540          107
16             P   R16     2152          269
17             P    QF     1076          269
18             P    SF      540          270
19             P     F      269          269
20             P  R128      192            6
21        

What the tables show
Grand Slams are clean: 65 events, with 64 first-round matches each (4,160 ÷ 65).
Events come in several formats, and the first round tells you which:
first round R128: two-week events (Indian Wells, Miami, Madrid-style)
first round R64: longer one-week events (a 56 or 64 draw, with byes)
first round R32: ordinary one-week events
Round robin (RR): four level-I events (96 matches) and one level-P event (12 matches). The four I events are almost certainly the United Cup, a team event we already excluded from the venue list, but check before assuming. The P one is probably the Elite Trophy.
The approach: a window of possible days, not a single day

We only know the tournament's start date and each match's round. A first-round match in a Grand Slam could be on day 0 or day 1, so claiming "day 0" would be false precision. Each match gets a range (match_date_min to match_date_max). The weather feature later averages daytime weather over that range, so the uncertainty carries through honestly. The ranges come from typical schedules, which are my assumptions and not data. We'll test them against known dates before relying on them.

What the checks show
Check 1: the format counts add up to 872 events and 38,860 matches, which is 3 fewer than before. Those are the leftover Zhuhai knockout rows removed by the new rule. Every match has a window (no_window is 0).
Check 2: 838 of 872 events start on a Monday, and 16 start on a Sunday. The other 18 are mid-week starts (probably the 2020 and 2021 schedule changes). The windows are measured from the recorded start date, so they stay broadly right, but they could be off by about a day for those 34. That's a limitation to log, and weather averaged over a window will smooth most of it.
Check 3: the real 2019 women's finals (26 Jan, 8 Jun, 13 Jul, 7 Sep) all fall inside their windows. That's a modest test, since it's four matches in the same round, so it's a sanity check and not a validation. A stronger test would use the Match Charting Project's real dates, if you get time for it.
Check 4: 6 events have match windows that extend past the weather we fetched. As predicted, these should be the two-week level-P events (the 6 tournaments with a 128 draw), which I fetched for only 8 days because I set the window by level, not by event format.